In [ ]:
# ============================================================
# Core imports + config
# ============================================================
import os, sys, gc
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from scipy.ndimage import gaussian_filter
from scipy.integrate import trapezoid
import seaborn as sns
import pickle
from tqdm.auto import tqdm

dt = 0.005
t_pre, t_post = 0.3, 0.3

# ── Dataset selection ──
SUBJECT = "HERCULE"           # subject name prefix for pickle files
HEADSTAGES = ["hs_0"]          # headstage suffixes to load and merge
SESSION_TYPE = "playback"      # "playback" or "eTheremin" session type
SPIKE_SORTED = True            # True = load _ss pickle files, False = unsorted

USE_MACRO_EPOCH = True   # True = contiguous Condition blocks, False = peri-event windows
CEBRA_DIM = 3             # CEBRA embedding dimension (2 or 3)
CEBRA_DISTANCE = "euclidean"  # "cosine" or "euclidean"
CEBRA_ARCH = "offset10-model" if CEBRA_DISTANCE == "cosine" else "offset10-model-mse"
TAU_SHIFT = 6       # label lag in bins (6 = 30ms at dt=0.005, 0 = disabled)
LIE_METHOD = "lstsq"  # "lstsq" = OLS + skew-symmetrize (recommended); "pytorch" = constrained optimization
MIN_EPOCH_DUR = 2.0      # minimum macro-epoch duration (seconds)

NAS = r"\\129.199.81.18\data6\eTheremin"

# Build file prefixes from SUBJECT + HEADSTAGES
FILE_PREFIXES = [f"{SUBJECT}_{hs}" for hs in HEADSTAGES]
HS_TAG = "_".join(HEADSTAGES)  # for filenames
PKL_SUFFIX = "_ss" if SPIKE_SORTED else ""  # pickle file suffix

mpl.rcdefaults()
plt.rcParams.update({
    'font.size': 7, 'axes.linewidth': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
    'xtick.major.size': 2, 'ytick.major.size': 2,
    'xtick.direction': 'out', 'ytick.direction': 'out',
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})

print(f"Subject: {SUBJECT}, Headstages: {HEADSTAGES}")
print(f"Spike sorted: {SPIKE_SORTED} (suffix='{PKL_SUFFIX}')")
print(f"CEBRA: {CEBRA_DIM}D, distance={CEBRA_DISTANCE}, arch={CEBRA_ARCH}")
print(f"NAS: {NAS}")
print("Imports ready.")

In [ ]:
# ============================================================
# Load spike-sorted data for selected subject + headstages
# ============================================================

def load_pickled(file_prefix, session_type, dt):
    """Load pickled neural + feature data. Suffix controlled by SPIKE_SORTED."""
    data_path = os.path.join(NAS, f"{file_prefix}_{session_type}_{dt}_data{PKL_SUFFIX}")
    feat_path = os.path.join(NAS, f"{file_prefix}_{session_type}_{dt}_feature{PKL_SUFFIX}")
    with open(data_path, "rb") as f: n_data = pickle.load(f)
    with open(feat_path, "rb") as f: f_data = pickle.load(f)
    return n_data, f_data

# Load all headstages dynamically
n_data_list_by_hs = []
f_data_list_by_hs = []
n_per_hs = []

for prefix in FILE_PREFIXES:
    print(f"Loading {prefix}...")
    n_data, f_data = load_pickled(prefix, SESSION_TYPE, dt)
    print(f"  {prefix}: {len(n_data)} sessions")
    n_data_list_by_hs.append(n_data)
    f_data_list_by_hs.append(f_data)
    n_per_hs.append(len(n_data))

# Add Velocity_x
for f_list in f_data_list_by_hs:
    for f_df in f_list:
        pos = f_df["Position"].values
        vel = np.diff(pos); vel = np.append(0, vel)
        vel = vel * 100; vel[~np.isfinite(vel)] = 0
        f_df["Velocity_x"] = vel

# Flatten: all sessions from all headstages
n_data_all_raw = []
f_data_all_raw = []
for n_list, f_list in zip(n_data_list_by_hs, f_data_list_by_hs):
    n_data_all_raw.extend(n_list)
    f_data_all_raw.extend(f_list)

# n_hs0 = number of sessions from first headstage
n_hs0 = n_per_hs[0]

MIN_GC = 10
n_data_all, f_data_all = [], []
n_hs0_filtered = 0
for i, nd in enumerate(n_data_all_raw):
    if nd.shape[0] >= MIN_GC:
        n_data_all.append(nd)
        f_data_all.append(f_data_all_raw[i])
        if i < n_hs0: n_hs0_filtered += 1

n_hs0 = n_hs0_filtered
print(f"After gc>={MIN_GC}: {len(n_data_all)} sessions ("
      + "/".join(f"{HEADSTAGES[j]}={n_per_hs[j]}" for j in range(len(HEADSTAGES))) + ")")

example = f_data_all[0]
print(f"Example: {example.shape[0]:,} tp, {n_data_all[0].shape[0]} neurons")
print(f"  Velocity_x: [{example['Velocity_x'].min():.1f}, {example['Velocity_x'].max():.1f}]")
print(f"  Position:   [{example['Position'].min():.1f}, {example['Position'].max():.1f}]")

In [ ]:
# ============================================================
# Helper functions: macro-epoch + unified extraction + Lie Algebra
# ============================================================


def preprocess_data(data_list, method="l2"):
    """Preprocess list of (time, neurons) arrays.
    method='l2': per-timepoint L2 normalization (for cosine distance)
    method='zscore': per-neuron Z-score across time (for euclidean distance)
    """
    out = []
    for d in data_list:
        if method == "zscore":
            mean = np.mean(d, axis=0, keepdims=True)
            std = np.std(d, axis=0, keepdims=True)
            std[std == 0] = 1e-9
            out.append(((d - mean) / std).astype(np.float32))
        else:  # l2
            norms = np.linalg.norm(d, axis=1, keepdims=True)
            norms[norms == 0] = 1e-9
            out.append((d / norms).astype(np.float32))
    return out


def extract_macro_epochs(n_data_session, f_df, condition_val, dt,
                          min_duration=2.0, label_col="Velocity_x"):
    """Extract contiguous macro-epochs of the same Condition."""
    conditions = f_df["Condition"].values
    mask = (conditions == condition_val)
    min_bins = int(min_duration / dt)
    epochs_n, epochs_l = [], []
    in_epoch, start = False, 0
    for i in range(len(mask)):
        if mask[i] and not in_epoch:
            start = i; in_epoch = True
        elif not mask[i] and in_epoch:
            if i - start >= min_bins:
                epochs_n.append(n_data_session[:, start:i].T.astype(np.float32))
                epochs_l.append(f_df[label_col].values[start:i].astype(np.float32))
            in_epoch = False
    if in_epoch and (len(mask) - start) >= min_bins:
        epochs_n.append(n_data_session[:, start:].T.astype(np.float32))
        epochs_l.append(f_df[label_col].values[start:].astype(np.float32))
    return epochs_n, epochs_l, len(epochs_n)


def extract_epochs(n_data_session, f_df, condition_val, dt,
                   label_col="Velocity_x", step=1):
    """Unified extraction: macro-epochs or peri-event windows."""
    if USE_MACRO_EPOCH:
        epochs_n, epochs_l, n_ep = extract_macro_epochs(
            n_data_session, f_df, condition_val, dt,
            min_duration=MIN_EPOCH_DUR, label_col=label_col)
        if step > 1:
            epochs_n = [e[::step].astype(np.float32) for e in epochs_n]
            epochs_l = [l[::step].astype(np.float32) for l in epochs_l]
        method = "l2" if CEBRA_DISTANCE == "cosine" else "zscore"
        epochs_n = preprocess_data(epochs_n, method=method)
        if TAU_SHIFT > 0:
            epochs_n, epochs_l = zip(*[(e_n[TAU_SHIFT:], e_l[:-TAU_SHIFT])
                                        for e_n, e_l in zip(epochs_n, epochs_l)])
            epochs_n, epochs_l = list(epochs_n), list(epochs_l)
        return epochs_n, epochs_l, n_ep
    else:
        trigger_mask = (f_df["Condition"].values == condition_val) & (f_df["Frequency_changes"].values == 1)
        trigger_indices = np.where(trigger_mask)[0]
        n_pre = int(t_pre / dt)
        n_post = int(t_post / dt)
        windows_n, windows_l = [], []
        for idx in trigger_indices:
            start = idx - n_pre
            end = idx + n_post + 1
            if start >= 0 and end <= n_data_session.shape[1]:
                windows_n.append(n_data_session[:, start:end].T.astype(np.float32))
                windows_l.append(f_df[label_col].values[start:end].astype(np.float32))
        if step > 1:
            windows_n = [w[::step].astype(np.float32) for w in windows_n]
            windows_l = [l[::step].astype(np.float32) for l in windows_l]
        method = "l2" if CEBRA_DISTANCE == "cosine" else "zscore"
        windows_n = preprocess_data(windows_n, method=method)
        if TAU_SHIFT > 0:
            windows_n, windows_l = zip(*[(w_n[TAU_SHIFT:], w_l[:-TAU_SHIFT])
                                          for w_n, w_l in zip(windows_n, windows_l)])
            windows_n, windows_l = list(windows_n), list(windows_l)
        return windows_n, windows_l, len(windows_n)


def _fit_lie_lstsq(r, x_dot, dt=0.005):
    """OLS + skew-symmetrization (original method)."""
    T, N = r.shape
    dr_dt = np.gradient(r, dt, axis=0)
    U_rot = r * x_dot[:, np.newaxis]
    U = np.hstack([U_rot, r])
    weights_T, _, _, _ = np.linalg.lstsq(U, dr_dt, rcond=None)
    weights = weights_T.T
    J_ols = weights[:, :N]
    J_skew = 0.5 * (J_ols - J_ols.T)
    norm_total = np.linalg.norm(J_ols)
    norm_skew = np.linalg.norm(J_skew)
    sr = norm_skew / norm_total if norm_total > 1e-9 else 0
    dR_pred = U_rot @ J_skew.T + r @ weights[:, N:].T
    ss_res = np.sum((dr_dt - dR_pred) ** 2)
    ss_tot = np.sum((dr_dt - np.mean(dr_dt)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 1e-9 else 0
    return J_skew, sr, r2, J_ols


def _fit_lie_pytorch(r, x_dot, dt=0.005, n_iter=500, lr=1e-3):
    """Constrained optimization: J_skew = W - W^T via PyTorch."""
    import torch
    T, N = r.shape
    dr_dt = np.gradient(r, dt, axis=0)
    R_t = torch.tensor(r, dtype=torch.float32)
    X_t = torch.tensor(x_dot, dtype=torch.float32).reshape(-1, 1)
    dR_t = torch.tensor(dr_dt, dtype=torch.float32)
    W = torch.zeros(N, N, requires_grad=True)
    L_t = torch.zeros(N, N, requires_grad=True)
    opt = torch.optim.Adam([W, L_t], lr=lr)
    for _ in range(n_iter):
        opt.zero_grad()
        J_t = W - W.T
        dR_pred = (R_t * X_t) @ J_t.T + R_t @ L_t.T
        loss = torch.mean((dR_t - dR_pred) ** 2)
        loss.backward()
        opt.step()
    with torch.no_grad():
        J_skew = (W - W.T).numpy()
        L_np = L_t.numpy()
    U = np.hstack([r * x_dot[:, None], r])
    w_T, _, _, _ = np.linalg.lstsq(U, dr_dt, rcond=None)
    J_ols = w_T.T[:, :N]
    sr = np.linalg.norm(J_skew) / np.linalg.norm(J_ols) if np.linalg.norm(J_ols) > 1e-9 else 0
    dR_pred = (r * x_dot[:, None]) @ J_skew.T + r @ L_np.T
    ss_res = np.sum((dr_dt - dR_pred) ** 2)
    ss_tot = np.sum((dr_dt - np.mean(dr_dt)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 1e-9 else 0
    return J_skew, sr, r2, J_ols


def fit_lie_algebra_with_leak(r, x_dot, dt=0.005, n_iter=500, lr=1e-3):
    """Fit dR/dt = J_skew * R * x_dot + L * R."""
    if LIE_METHOD == "lstsq":
        return _fit_lie_lstsq(r, x_dot, dt)
    else:
        return _fit_lie_pytorch(r, x_dot, dt, n_iter, lr)


print("Helpers ready: extract_macro_epochs, extract_epochs, fit_lie_algebra_with_leak")

In [ ]:
# ============================================================
# Macro-Epoch / Peri-Event Extraction Report
# ============================================================
for label_col, driver_name in [("Velocity_x", "Velocity"), ("Position", "Position")]:
    print("=" * 50)
    print(f"  {driver_name}-Driven")
    print("=" * 50)
    for val, cond_name in [(0.0, "Tracking"), (1.0, "Playback")]:
        total_epochs, total_pts, total_sec = 0, 0, 0
        for idx, (n_data_session, f_df) in enumerate(zip(n_data_all, f_data_all)):
            epochs_n, epochs_l, n_ep = extract_epochs(
                n_data_session, f_df, val, dt, label_col=label_col)
            total_epochs += n_ep
            total_pts += sum(e.shape[0] for e in epochs_n)
            total_sec += sum(e.shape[0] for e in epochs_n) * dt
        mode = "macro-epoch" if USE_MACRO_EPOCH else "peri-event"
        print(f"  {cond_name}: {total_epochs} {mode}s, {total_pts:,} pts, {total_sec:.0f}s total")
    print()

print("Proceed to analysis cells.")

## 1. Lie Algebra Dynamics

Equation: $d\mathbf{R}/dt = \mathbf{J}_{\text{skew}} \cdot \mathbf{R} \cdot x + \mathbf{L} \cdot \mathbf{R}$

$\mathbf{J}_{\text{skew}}$ = rotation generator, $\mathbf{L}$ = leak/dissipation.

### 1.1 Velocity-Driven

In [ ]:
# ==========================================
# Lie Algebra Generator Analysis — Velocity-Driven
# ==========================================
all_results = []

for idx, (n_data_session, f_df) in enumerate(zip(n_data_all, f_data_all)):
    r_full = n_data_session.T
    x_dot_full = f_df["Velocity_x"].values
    conditions = f_df["Condition"].values
    hs_label = HEADSTAGES[0] if idx < n_hs0 else HEADSTAGES[-1] if len(HEADSTAGES) > 1 else HEADSTAGES[0]

    for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
        epochs_n, epochs_l, n_ep = extract_epochs(
            n_data_session, f_df, val, dt,
            label_col="Velocity_x")
        if n_ep < 1:
            continue
        r_sub = np.concatenate(epochs_n, axis=0)
        x_dot_sub = np.concatenate(epochs_l, axis=0)

        try:
            J_skew, skew_ratio, r2, J_full = fit_lie_algebra_with_leak(r_sub, x_dot_sub)
            all_results.append({
                "Subject": SUBJECT,
                "Session_Idx": idx,
                "Headstage": hs_label,
                "Condition": label,
                "Skewness_Ratio": skew_ratio,
                "R2": r2,
                "N_Neurons": r_sub.shape[1],
                "N_Timepoints": r_sub.shape[0],
            })
        except Exception as e:
            print(f"Error {SUBJECT} session {idx} [{label}]: {e}")

results_df = pd.DataFrame(all_results)
print()
print("Analysis complete: " + str(len(results_df)) + " entries")
print(results_df.groupby(["Headstage", "Condition"])[["Skewness_Ratio", "R2"]].mean().round(4))

In [ ]:
# ============================================================
# Velocity Lie Algebra — Visualization & Paired Statistics
# ============================================================
from scipy.stats import ttest_rel

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.barplot(data=results_df, x="Headstage", y="Skewness_Ratio", hue="Condition",
            ax=axes[0], palette="viridis", errorbar="se")
axes[0].set_title(f"{SUBJECT}: Skewness Ratio (Velocity-Driven)")

sns.barplot(data=results_df, x="Headstage", y="R2", hue="Condition",
            ax=axes[1], palette="magma", errorbar="se")
axes[1].set_title(f"{SUBJECT}: Model R" + chr(0x00B2))

pivot = results_df.pivot_table(
    values=["Skewness_Ratio", "R2"],
    index=["Subject", "Session_Idx", "Headstage"], columns="Condition"
).dropna()

if len(pivot) > 0:
    t_skew, p_skew = ttest_rel(pivot["Skewness_Ratio"]["Tracking"],
                               pivot["Skewness_Ratio"]["Playback"])

    tr_vals = pivot["Skewness_Ratio"]["Tracking"].values
    pb_vals = pivot["Skewness_Ratio"]["Playback"].values
    axes[2].scatter(tr_vals, pb_vals, alpha=0.6, c="steelblue")
    lims = [min(tr_vals.min(), pb_vals.min()) - 0.02,
            max(tr_vals.max(), pb_vals.max()) + 0.02]
    axes[2].plot(lims, lims, "--", color="gray", alpha=0.5, label="y = x")
    axes[2].set_xlabel("Tracking Skewness Ratio")
    axes[2].set_ylabel("Playback Skewness Ratio")
    axes[2].set_title(f"Paired (N={len(pivot)})\nt={t_skew:.2f}, p={p_skew:.4f}")
    axes[2].legend()

    print(f"Paired t-test (N={len(pivot)} pairs):")
    print(f"  Skewness Ratio: t={t_skew:.3f}, p={p_skew:.6f}")
    print(f"  Tracking  {tr_vals.mean():.4f} +/- {tr_vals.std():.4f}")
    print(f"  Playback  {pb_vals.mean():.4f} +/- {pb_vals.std():.4f}")
else:
    print("Not enough paired sessions for t-test.")

plt.suptitle(f"{SUBJECT}: Lie Algebra — Active Tracking vs Passive Playback (Velocity)",
             fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ==========================================
# Eigenvalue Stability Analysis — Velocity-Driven
# ==========================================
min_points = 500
eig_results = []

for idx, (n_data_session, f_df) in enumerate(zip(n_data_all, f_data_all)):
    r_full = n_data_session.T
    x_dot_full = f_df["Velocity_x"].values
    conditions = f_df["Condition"].values
    hs_label = HEADSTAGES[0] if idx < n_hs0 else HEADSTAGES[-1] if len(HEADSTAGES) > 1 else HEADSTAGES[0]

    for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
        epochs_n, epochs_l, n_ep = extract_epochs(
            n_data_session, f_df, val, dt,
            label_col="Velocity_x")
        if n_ep < 1:
            continue
        r_sub = np.concatenate(epochs_n, axis=0)
        x_dot_sub = np.concatenate(epochs_l, axis=0)

        try:
            _, _, _, J_raw = fit_lie_algebra_with_leak(r_sub, x_dot_sub)
            eigvals = np.linalg.eigvals(J_raw)
            real_mean = np.mean(np.abs(np.real(eigvals)))
            imag_mean = np.mean(np.abs(np.imag(eigvals)))
            eig_results.append({
                "Subject": SUBJECT, "Session_Idx": idx, "Headstage": hs_label,
                "Condition": label, "Real_Mean": real_mean,
                "Imag_Mean": imag_mean,
                "Imag_Real_Ratio": imag_mean / real_mean if real_mean > 1e-9 else 0,
                "N_Neurons": r_full.shape[1],
            })
        except Exception as e:
            print(f"Eig error {SUBJECT} {idx} [{label}]: {e}")

eig_df = pd.DataFrame(eig_results)

pivot_eig = eig_df.pivot_table(
    values=["Real_Mean", "Imag_Mean"],
    index=["Subject", "Session_Idx", "Headstage"],
    columns="Condition"
).dropna()

if len(pivot_eig) > 0:
    t_real, p_real = ttest_rel(pivot_eig["Real_Mean"]["Tracking"], pivot_eig["Real_Mean"]["Playback"])
    t_imag, p_imag = ttest_rel(pivot_eig["Imag_Mean"]["Tracking"], pivot_eig["Imag_Mean"]["Playback"])

    print("Eigenvalue paired t-test (N=" + str(len(pivot_eig)) + "):")
    print(f"  |Real| (Dissipation): t={t_real:.3f}, p={p_real:.6f}")
    print(f"  |Imag| (Rotation):    t={t_imag:.3f}, p={p_imag:.6f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    configs = [
        (axes[0], "Real_Mean", "Dissipation (|Real|)\np=" + str(round(p_real, 4)), "red"),
        (axes[1], "Imag_Mean", "Rotation (|Imag|)\np=" + str(round(p_imag, 4)), "blue"),
    ]
    for ax, col, ttl, color in configs:
        for _, row in pivot_eig.iterrows():
            ax.plot([0, 1], [row[col]["Tracking"], row[col]["Playback"]],
                    "o-", color="gray", alpha=0.3, markersize=4, linewidth=0.8)
        ax.plot([0, 1], [pivot_eig[col]["Tracking"].mean(), pivot_eig[col]["Playback"].mean()],
                "o-", color=color, linewidth=3, markersize=10, label="Mean")
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["Tracking", "Playback"])
        ax.set_ylabel(col)
        ax.set_title(ttl)
        ax.legend()
    plt.suptitle(f"{SUBJECT}: Dynamical Stability Signatures", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough paired sessions for eigenvalue analysis.")

### 1.2 Position-Driven

In [ ]:
# ==========================================
# Position-Driven Lie Algebra
# ==========================================
pos_results = []

for idx, (n_data_session, f_df) in enumerate(zip(n_data_all, f_data_all)):
    r_full = n_data_session.T
    x_full = f_df["Position"].values
    conditions = f_df["Condition"].values
    hs_label = HEADSTAGES[0] if idx < n_hs0 else HEADSTAGES[-1] if len(HEADSTAGES) > 1 else HEADSTAGES[0]

    for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
        epochs_n, epochs_l, n_ep = extract_epochs(
            n_data_session, f_df, val, dt,
            label_col="Position")
        if n_ep < 1:
            continue
        r_sub = np.concatenate(epochs_n, axis=0)
        x_sub = np.concatenate(epochs_l, axis=0)

        try:
            J_skew, skew_ratio, r2, J_full = fit_lie_algebra_with_leak(r_sub, x_sub)
            pos_results.append({
                "Subject": SUBJECT,
                "Session_Idx": idx,
                "Headstage": hs_label,
                "Condition": label,
                "Skewness_Ratio": skew_ratio,
                "R2": r2,
                "N_Neurons": r_sub.shape[1],
            })
        except Exception as e:
            print(f"Error {SUBJECT} {idx} [{label}]: {e}")

pos_df = pd.DataFrame(pos_results)
print("Position-driven analysis: " + str(len(pos_df)) + " entries")
print(pos_df.groupby(["Headstage", "Condition"])[["Skewness_Ratio", "R2"]].mean().round(4))

# ---- Comparison: Velocity vs Position ----
print()
print("=" * 65)
print("  Velocity-driven vs Position-driven Skewness Ratio")
print("=" * 65)
print()
print("  Driver       Condition     Skewness (mean)    R2 (mean)")
print("  " + "-" * 55)
for driver, df in [("Velocity", results_df), ("Position", pos_df)]:
    for cond in ["Tracking", "Playback"]:
        subset = df[df["Condition"] == cond]
        if len(subset) > 0:
            sk = subset["Skewness_Ratio"].mean()
            r2 = subset["R2"].mean()
            print("  {:<12s} {:<12s}  {:.4f}            {:.4f}".format(driver, cond, sk, r2))

# Paired t-test for position-driven
pivot_pos = pos_df.pivot_table(
    values=["Skewness_Ratio", "R2"],
    index=["Subject", "Session_Idx", "Headstage"],
    columns="Condition"
).dropna()

if len(pivot_pos) > 0:
    t_pos, p_pos = ttest_rel(pivot_pos["Skewness_Ratio"]["Tracking"],
                              pivot_pos["Skewness_Ratio"]["Playback"])
    print()
    print(f"Position-driven paired t-test (N={len(pivot_pos)}):")
    print(f"  t={t_pos:.3f}, p={p_pos:.6f}")

# Barplot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=pos_df, x="Headstage", y="Skewness_Ratio", hue="Condition",
            ax=axes[0], palette="viridis", errorbar="se")
axes[0].set_title(f"{SUBJECT}: Position-Driven Skewness Ratio")
axes[0].set_ylabel("Skewness Ratio")

drivers = []
for driver, df in [("Velocity", results_df), ("Position", pos_df)]:
    for _, row in df.iterrows():
        drivers.append({"Driver": driver, "Condition": row["Condition"],
                        "Skewness_Ratio": row["Skewness_Ratio"]})
driver_df = pd.DataFrame(drivers)
sns.barplot(data=driver_df, x="Driver", y="Skewness_Ratio", hue="Condition",
            ax=axes[1], palette="viridis", errorbar="se")
axes[1].set_title("Velocity vs Position Driver")
axes[1].set_ylabel("Skewness Ratio")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Position Eigenvalue Stability Analysis
# ============================================================
eig_results_pos = []

for idx, (n_data_session, f_df) in enumerate(zip(n_data_all, f_data_all)):
    r_full = n_data_session.T
    x_full = f_df["Position"].values
    conditions = f_df["Condition"].values
    hs_label = HEADSTAGES[0] if idx < n_hs0 else HEADSTAGES[-1] if len(HEADSTAGES) > 1 else HEADSTAGES[0]

    for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
        epochs_n, epochs_l, n_ep = extract_epochs(
            n_data_session, f_df, val, dt,
            label_col="Position")
        if n_ep < 1:
            continue
        r_sub = np.concatenate(epochs_n, axis=0)
        x_sub = np.concatenate(epochs_l, axis=0)

        try:
            _, _, _, J_raw = fit_lie_algebra_with_leak(r_sub, x_sub)
            eigvals = np.linalg.eigvals(J_raw)
            real_mean = np.mean(np.abs(np.real(eigvals)))
            imag_mean = np.mean(np.abs(np.imag(eigvals)))
            eig_results_pos.append({
                "Subject": SUBJECT, "Session_Idx": idx, "Headstage": hs_label,
                "Condition": label,
                "Real_Mean": real_mean, "Imag_Mean": imag_mean,
                "Imag_Real_Ratio": imag_mean / real_mean if real_mean > 1e-9 else 0,
                "N_Neurons": r_full.shape[1],
            })
        except Exception as e:
            print(f"Eig error {SUBJECT} {idx} [{label}]: {e}")

eig_pos_df = pd.DataFrame(eig_results_pos)

pivot_eig_pos = eig_pos_df.pivot_table(
    values=["Real_Mean", "Imag_Mean"],
    index=["Subject", "Session_Idx", "Headstage"], columns="Condition"
).dropna()

if len(pivot_eig_pos) > 0:
    t_real_p, p_real_p = ttest_rel(pivot_eig_pos["Real_Mean"]["Tracking"],
                                    pivot_eig_pos["Real_Mean"]["Playback"])
    t_imag_p, p_imag_p = ttest_rel(pivot_eig_pos["Imag_Mean"]["Tracking"],
                                    pivot_eig_pos["Imag_Mean"]["Playback"])

    print(f"Position Eigenvalue paired t-test (N={len(pivot_eig_pos)}):")
    print(f"  |Real| (Dissipation): t={t_real_p:.3f}, p={p_real_p:.6f}")
    print(f"  |Imag| (Rotation):    t={t_imag_p:.3f}, p={p_imag_p:.6f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    configs = [
        (axes[0], "Real_Mean", "Dissipation (|Real|)\np=" + str(round(p_real_p, 4)), "red"),
        (axes[1], "Imag_Mean", "Rotation (|Imag|)\np=" + str(round(p_imag_p, 4)), "blue"),
    ]
    for ax, col, ttl, color in configs:
        for _, row in pivot_eig_pos.iterrows():
            ax.plot([0, 1], [row[col]["Tracking"], row[col]["Playback"]],
                    "o-", color="gray", alpha=0.3, markersize=4, linewidth=0.8)
        ax.plot([0, 1], [pivot_eig_pos[col]["Tracking"].mean(), pivot_eig_pos[col]["Playback"].mean()],
                "o-", color=color, linewidth=3, markersize=10, label="Mean")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["Tracking", "Playback"])
        ax.set_ylabel(col); ax.set_title(ttl); ax.legend()
    plt.suptitle(f"{SUBJECT}: Dynamical Stability — Position-Driven", fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()
else:
    print("Not enough paired sessions for position eigenvalue analysis.")

### 1.3 CEBRA-Embedded Lie Algebra (exploratory)

**Caveat:** CEBRA is trained with the same behavioral variable (Velocity/Position) used as the Lie algebra drive — so a high skewness ratio may partly reflect the supervised embedding objective rather than intrinsic rotational dynamics. This section tests whether nonlinear dimensionality reduction makes rotational structure more apparent, but does **not** constitute topological evidence (cf. Gardner 2022, Hermansen 2024).

**Design:**
- One pooled CEBRA model per session (Tracking + Playback share coordinate system)
- Lie algebra fit **per epoch** (no concatenation → no derivative jumps at boundaries)
- **Time-shuffled control:** 10 random label permutations per epoch → baseline skewness/R²
- **R² gate:** skewness ratio is only treated as meaningful when true R² exceeds shuffle R²
- Per-condition statistics reported separately (Tracking and Playback share CEBRA model)

**Remaining limitations:**
- *Shuffle-CEBRA control:* the strongest negative control would train CEBRA on shuffled labels and then fit Lie algebra — this is not done here (computational cost, exploratory scope)
- *Session embeddings not comparable:* each session has its own latent coordinate system; interpret only within-session True-vs-Shuffle differences, not cross-session absolute values
- *Dimensionality:* results may depend on CEBRA embedding dimension (3D here); toroidal structure requires testing across 2D/3D/6D

In [ ]:
# ============================================================
# CEBRA-Embedded Lie Algebra — Velocity-Driven (per-epoch + shuffle)
# ============================================================
#   a) Train ONE pooled CEBRA per session
#   b) Per epoch: transform → fit Lie algebra → (sr, r2)
#   c) Per epoch: shuffle labels → fit Lie algebra → (sr_shuf, r2_shuf)
#   d) Aggregate across epochs → paired True vs Shuffle comparison

try:
    from cebra import CEBRA
    HAS_CEBRA_LIE = True
except ImportError:
    HAS_CEBRA_LIE = False
    print("CEBRA not installed. Skipping CEBRA-embedded Lie algebra.")

CEBRA_EMBEDDING_DIM = 3
CEBRA_LIE_ITERS = 2000
MIN_EPOCH_TIMEPOINTS = 200
MIN_EPOCHS_PER_COND = 1
N_SHUFFLES = 10           # shuffle realizations per epoch (reduces baseline noise)

if HAS_CEBRA_LIE:
    import torch
    cebra_lie_vel = []      # per-condition, per-session aggregate
    epoch_results_vel = []  # per-epoch (for within-session SEM)
    cebra_losses = []
    n_skipped_epochs, n_skipped_cond, n_sessions_used = 0, 0, 0

    for idx, (n_data_session, f_df) in enumerate(zip(n_data_all, f_data_all)):
        hs_label = HEADSTAGES[0] if idx < n_hs0 else HEADSTAGES[-1] if len(HEADSTAGES) > 1 else HEADSTAGES[0]

        # ---- Extract + filter epochs for BOTH conditions ----
        cond_epochs = {}; cond_labels_raw = {}; skip_session = False
        for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
            epochs_n, epochs_l, n_ep = extract_epochs(
                n_data_session, f_df, val, dt, label_col="Velocity_x")
            valid_pairs = [(e, l) for e, l in zip(epochs_n, epochs_l)
                           if e.shape[0] >= MIN_EPOCH_TIMEPOINTS]
            n_skipped_epochs += len(epochs_n) - len(valid_pairs)
            if len(valid_pairs) < MIN_EPOCHS_PER_COND:
                skip_session = True; n_skipped_cond += 1; break
            cond_epochs[label] = [e for e, l in valid_pairs]
            cond_labels_raw[label] = [l for e, l in valid_pairs]
        if skip_session: continue
        n_sessions_used += 1

        # ---- Train ONE pooled CEBRA ----
        all_epochs = cond_epochs["Tracking"] + cond_epochs["Playback"]
        all_labels = cond_labels_raw["Tracking"] + cond_labels_raw["Playback"]
        n_track_ep = len(cond_epochs["Tracking"])

        try:
            cebra_model = CEBRA(
                model_architecture=CEBRA_ARCH, output_dimension=CEBRA_EMBEDDING_DIM,
                max_iterations=CEBRA_LIE_ITERS, batch_size=512, learning_rate=3e-4,
                temperature=1.5, distance=CEBRA_DISTANCE,
                conditional="time_delta", device="cuda", verbose=False)
            cebra_model.fit(all_epochs, all_labels)
            cebra_losses.append({"Session_Idx": idx, "Headstage": hs_label,
                                 "final_loss": float(cebra_model.state_dict_['loss'][-1]),
                                 "n_epochs": len(all_epochs)})

            # ---- Per-condition: fit Lie per epoch ----
            for label, start_idx in [("Tracking", 0), ("Playback", n_track_ep)]:
                epochs_n = cond_epochs[label]
                epochs_l = cond_labels_raw[label]
                sr_list, r2_list, sr_shuf_list, r2_shuf_list = [], [], [], []

                for i, (e_n, e_l) in enumerate(zip(epochs_n, epochs_l)):
                    emb = cebra_model.transform(all_epochs[start_idx + i], session_id=start_idx + i)

                    # True fit
                    J_s, sr, r2, _ = fit_lie_algebra_with_leak(emb, e_l)
                    sr_list.append(sr); r2_list.append(r2)

                    # Shuffled control: N_SHUFFLES realizations, averaged
                    sr_sh_vals, r2_sh_vals = [], []
                    for _ in range(N_SHUFFLES):
                        e_l_shuf = np.random.permutation(e_l)
                        _, sr_sh, r2_sh, _ = fit_lie_algebra_with_leak(emb, e_l_shuf)
                        sr_sh_vals.append(sr_sh); r2_sh_vals.append(r2_sh)
                    sr_shuf_list.append(np.mean(sr_sh_vals))
                    r2_shuf_list.append(np.mean(r2_sh_vals))

                # Aggregate across epochs for this session-condition
                cebra_lie_vel.append({
                    "Subject": SUBJECT, "Session_Idx": idx, "Headstage": hs_label,
                    "Condition": label, "Space": "CEBRA",
                    "Skewness_Ratio": np.mean(sr_list), "R2": np.mean(r2_list),
                    "Skewness_Ratio_SEM": np.std(sr_list) / max(1, np.sqrt(len(sr_list))),
                    "R2_SEM": np.std(r2_list) / max(1, np.sqrt(len(r2_list))),
                    "SR_shuffle": np.mean(sr_shuf_list), "R2_shuffle": np.mean(r2_shuf_list),
                    "N_Epochs": len(sr_list), "N_Dims": CEBRA_EMBEDDING_DIM,
                    "N_Neurons": n_data_session.shape[0],
                })

            del cebra_model; torch.cuda.empty_cache()
        except Exception as e:
            print(f"CEBRA-Lie err {SUBJECT} s{idx}: {e}")

    # ---- Report ----
    print(f"Sessions used: {n_sessions_used}/{len(n_data_all)}")
    print(f"  Skipped (<{MIN_EPOCHS_PER_COND} epoch/cond): {n_skipped_cond}")
    print(f"  Epochs dropped (<{MIN_EPOCH_TIMEPOINTS} tp): {n_skipped_epochs}")
    cebra_lie_vel_df = pd.DataFrame(cebra_lie_vel)

    if len(cebra_lie_vel_df) == 0:
        print("No CEBRA-embedded Lie results.")
    else:
        # Convergence check
        losses_df = pd.DataFrame(cebra_losses)
        print(f"\nCEBRA convergence ({CEBRA_LIE_ITERS} iters): "
              f"loss={losses_df['final_loss'].mean():.4f} +/- {losses_df['final_loss'].std():.4f}")

        # ---- True vs Shuffle comparison (THE KEY RESULT) ----
        print()
        print("=" * 70)
        print("  CEBRA-Embedded Lie: True vs Time-Shuffled Control (Velocity)")
        print("=" * 70)
        print(f"  {'Condition':<12} {'SR_true':>10} {'SR_shuf':>10} {'Delta':>10} "
              f"{'R2_true':>10} {'R2_shuf':>10} {'Delta':>10}")
        print(f"  {'-'*62}")
        for cond in ["Tracking", "Playback"]:
            sub = cebra_lie_vel_df[cebra_lie_vel_df["Condition"] == cond]
            sr_t, sr_s = sub["Skewness_Ratio"].mean(), sub["SR_shuffle"].mean()
            r2_t, r2_s = sub["R2"].mean(), sub["R2_shuffle"].mean()
            dsr = sr_t - sr_s
            dr2 = r2_t - r2_s
            sig_sr = " ***" if dsr > 0.05 else ""
            print(f"  {cond:<12} {sr_t:>10.4f} {sr_s:>10.4f} {dsr:>+10.4f}{sig_sr} "
                  f"{r2_t:>10.5f} {r2_s:>10.5f} {dr2:>+10.5f}")

        # Paired t-test: True vs Shuffle per condition (separate tests —
        # Tracking and Playback share a CEBRA model so pooling violates independence)
        print(f"\n  True-vs-Shuffle paired t-test (per condition, N={N_SHUFFLES} shuffles/epoch):")
        for cond in ["Tracking", "Playback"]:
            sub = cebra_lie_vel_df[cebra_lie_vel_df["Condition"] == cond]
            if len(sub) > 1:
                t_sr, p_sr = ttest_rel(sub["Skewness_Ratio"].values, sub["SR_shuffle"].values)
                t_r2, p_r2 = ttest_rel(sub["R2"].values, sub["R2_shuffle"].values)
                print(f"    {cond:<12}: SR true={sub['Skewness_Ratio'].mean():.4f} "
                      f"shuf={sub['SR_shuffle'].mean():.4f}, t={t_sr:.3f}, p={p_sr:.6f}")
                print(f"                R² true={sub['R2'].mean():.5f} "
                      f"shuf={sub['R2_shuffle'].mean():.5f}, t={t_r2:.3f}, p={p_r2:.6f}")

        # ---- R² gate: only meaningful if true R² > shuffle R² ----
        print()
        print("  R² gate check (skewness meaningful only when R²_true > R²_shuffle):")
        for cond in ["Tracking", "Playback"]:
            sub = cebra_lie_vel_df[cebra_lie_vel_df["Condition"] == cond]
            n_pass = (sub["R2"] > sub["R2_shuffle"]).sum()
            n_total = len(sub)
            sr_pass = sub[sub["R2"] > sub["R2_shuffle"]]["Skewness_Ratio"].mean() if n_pass > 0 else float('nan')
            sr_all = sub["Skewness_Ratio"].mean()
            print(f"    {cond:<12}: {n_pass}/{n_total} sessions pass R² gate "
                  f"(SR_pass={sr_pass:.4f}, SR_all={sr_all:.4f})")
        print("    ↑ If R²_true ≤ R²_shuffle, the Lie generator does not explain the trajectory")
        print("      derivative — high skewness in those cases is noise, not signal.")
        # Also report at session level: average Tracking+PB per session
        pv = cebra_lie_vel_df.pivot_table(
            values=["Skewness_Ratio", "SR_shuffle", "R2", "R2_shuffle"],
            index=["Subject", "Session_Idx", "Headstage"]).dropna()
        if len(pv) > 0:
            # Average across conditions within each session
            t_sr_sess, p_sr_sess = ttest_rel(
                pv["Skewness_Ratio"].mean(axis=1), pv["SR_shuffle"].mean(axis=1))
            print(f"    Session-level (T+PB avg, N={len(pv)}): "
                  f"SR t={t_sr_sess:.3f}, p={p_sr_sess:.6f}")

        # ---- Visualization ----
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))

        # Panel A: Skewness Ratio True vs Shuffle
        melted_sr = []
        for _, row in cebra_lie_vel_df.iterrows():
            melted_sr.append({"Condition": row["Condition"], "Type": "True",
                              "Skewness_Ratio": row["Skewness_Ratio"]})
            melted_sr.append({"Condition": row["Condition"], "Type": "Shuffle",
                              "Skewness_Ratio": row["SR_shuffle"]})
        sr_melt = pd.DataFrame(melted_sr)
        sns.barplot(data=sr_melt, x="Condition", y="Skewness_Ratio", hue="Type",
                    ax=axes[0], palette={"True": "#440154", "Shuffle": "#B2B2B2"}, errorbar="se")
        axes[0].set_title("Skewness Ratio: True vs Shuffle (per-epoch mean ± SEM)" if len(cebra_lie_vel_df) > 0
                          else "Skewness Ratio: True vs Shuffle")
        axes[0].set_ylabel("Skewness Ratio")

        # Panel B: R² True vs Shuffle
        melted_r2 = []
        for _, row in cebra_lie_vel_df.iterrows():
            melted_r2.append({"Condition": row["Condition"], "Type": "True", "R2": row["R2"]})
            melted_r2.append({"Condition": row["Condition"], "Type": "Shuffle", "R2": row["R2_shuffle"]})
        r2_melt = pd.DataFrame(melted_r2)
        sns.barplot(data=r2_melt, x="Condition", y="R2", hue="Type",
                    ax=axes[1], palette={"True": "#440154", "Shuffle": "#B2B2B2"}, errorbar="se")
        axes[1].set_title("R²: True vs Shuffle (per-epoch mean ± SEM)" if len(cebra_lie_vel_df) > 0
                          else "R²: True vs Shuffle")
        axes[1].set_ylabel("R²")

        # Panel C: Track-vs-PB in CEBRA space (paired)
        if len(pv) > 0:
            tr_sr = pv["Skewness_Ratio"]["Tracking"].values
            pb_sr = pv["Skewness_Ratio"]["Playback"].values
            t_trpb, p_trpb = ttest_rel(tr_sr, pb_sr)
            axes[2].scatter(tr_sr, pb_sr, alpha=0.6, c="steelblue")
            lims = [min(tr_sr.min(), pb_sr.min()) - 0.02,
                    max(tr_sr.max(), pb_sr.max()) + 0.02]
            axes[2].plot(lims, lims, "--", color="gray", alpha=0.5, label="y=x")
            axes[2].set_xlabel("Tracking SR"); axes[2].set_ylabel("Playback SR")
            axes[2].set_title(f"CEBRA Space Track vs PB\nN={len(pv)}, t={t_trpb:.2f}, p={p_trpb:.4f}")
            axes[2].legend()

        plt.suptitle(f"{SUBJECT}: CEBRA-Embedded Lie Algebra — Velocity-Driven (per-epoch + shuffle)",
                     fontsize=13, y=1.02)
        plt.tight_layout(); plt.show()

else:
    print("Skipped: CEBRA not installed.")

In [ ]:
# ============================================================
# CEBRA-Embedded Lie Algebra -- Position-Driven (per-epoch + shuffle)
# ============================================================

if HAS_CEBRA_LIE:
    import torch
    cebra_lie_pos = []
    cebra_losses_pos = []
    n_skipped_epochs, n_skipped_cond, n_sessions_used = 0, 0, 0
    N_SHUFFLES = 10           # shuffle realizations per epoch

    for idx, (n_data_session, f_df) in enumerate(zip(n_data_all, f_data_all)):
        hs_label = HEADSTAGES[0] if idx < n_hs0 else HEADSTAGES[-1] if len(HEADSTAGES) > 1 else HEADSTAGES[0]

        cond_epochs = {}; cond_labels_raw = {}; skip_session = False
        for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
            epochs_n, epochs_l, n_ep = extract_epochs(
                n_data_session, f_df, val, dt, label_col="Position")
            valid_pairs = [(e, l) for e, l in zip(epochs_n, epochs_l)
                           if e.shape[0] >= MIN_EPOCH_TIMEPOINTS]
            n_skipped_epochs += len(epochs_n) - len(valid_pairs)
            if len(valid_pairs) < MIN_EPOCHS_PER_COND:
                skip_session = True; n_skipped_cond += 1; break
            cond_epochs[label] = [e for e, l in valid_pairs]
            cond_labels_raw[label] = [l for e, l in valid_pairs]
        if skip_session: continue
        n_sessions_used += 1

        all_epochs = cond_epochs["Tracking"] + cond_epochs["Playback"]
        all_labels = cond_labels_raw["Tracking"] + cond_labels_raw["Playback"]
        n_track_ep = len(cond_epochs["Tracking"])

        try:
            cebra_model = CEBRA(
                model_architecture=CEBRA_ARCH, output_dimension=CEBRA_EMBEDDING_DIM,
                max_iterations=CEBRA_LIE_ITERS, batch_size=512, learning_rate=3e-4,
                temperature=1.5, distance=CEBRA_DISTANCE,
                conditional="time_delta", device="cuda", verbose=False)
            cebra_model.fit(all_epochs, all_labels)
            cebra_losses_pos.append({"Session_Idx": idx,
                                     "final_loss": float(cebra_model.state_dict_["loss"][-1])})

            for label, start_idx in [("Tracking", 0), ("Playback", n_track_ep)]:
                epochs_n = cond_epochs[label]
                epochs_l = cond_labels_raw[label]
                sr_list, r2_list, sr_shuf_list, r2_shuf_list = [], [], [], []

                for i, (e_n, e_l) in enumerate(zip(epochs_n, epochs_l)):
                    emb = cebra_model.transform(all_epochs[start_idx + i], session_id=start_idx + i)
                    # True
                    _, sr, r2, _ = fit_lie_algebra_with_leak(emb, e_l)
                    sr_list.append(sr); r2_list.append(r2)
                    # Shuffle: N_SHUFFLES realizations, averaged
                    sr_sh_vals, r2_sh_vals = [], []
                    for _ in range(N_SHUFFLES):
                        _, sr_sh, r2_sh, _ = fit_lie_algebra_with_leak(emb, np.random.permutation(e_l))
                        sr_sh_vals.append(sr_sh); r2_sh_vals.append(r2_sh)
                    sr_shuf_list.append(np.mean(sr_sh_vals))
                    r2_shuf_list.append(np.mean(r2_sh_vals))

                cebra_lie_pos.append({
                    "Subject": SUBJECT, "Session_Idx": idx, "Headstage": hs_label,
                    "Condition": label, "Space": "CEBRA",
                    "Skewness_Ratio": np.mean(sr_list), "R2": np.mean(r2_list),
                    "SR_shuffle": np.mean(sr_shuf_list), "R2_shuffle": np.mean(r2_shuf_list),
                    "N_Epochs": len(sr_list), "N_Dims": CEBRA_EMBEDDING_DIM,
                })
            del cebra_model; torch.cuda.empty_cache()
        except Exception as e:
            print(f"CEBRA-Lie-Pos err {SUBJECT} s{idx}: {e}")

    cebra_lie_pos_df = pd.DataFrame(cebra_lie_pos)
    if len(cebra_lie_pos_df) == 0:
        print("No CEBRA-embedded Lie results (Position).")
    else:
        print(f"Sessions used: {n_sessions_used}/{len(n_data_all)}")
        print(f"CEBRA convergence: loss={pd.DataFrame(cebra_losses_pos)["final_loss"].mean():.4f}")

        # True vs Shuffle
        print()
        print("=" * 70)
        print("  CEBRA-Embedded Lie: True vs Time-Shuffled Control (Position)")
        print("=" * 70)
        msg = "\n  True-vs-Shuffle paired t-test (per condition, N=" + str(N_SHUFFLES) + " shuffles/epoch):"
        print(msg)
        for cond in ["Tracking", "Playback"]:
            sub = cebra_lie_pos_df[cebra_lie_pos_df["Condition"] == cond]
            if len(sub) > 1:
                t_srp, p_srp = ttest_rel(sub["Skewness_Ratio"].values, sub["SR_shuffle"].values)
                print(f"    {cond:<12}: SR true={sub["Skewness_Ratio"].mean():.4f}, "
                      f"shuf={sub["SR_shuffle"].mean():.4f}, t={t_srp:.3f}, p={p_srp:.6f}")

        # R2 gate check
        print()
        print("  R2 gate check (skewness meaningful only when R2_true > R2_shuffle):")
        for cond in ["Tracking", "Playback"]:
            sub = cebra_lie_pos_df[cebra_lie_pos_df["Condition"] == cond]
            n_pass = (sub["R2"] > sub["R2_shuffle"]).sum()
            n_total = len(sub)
            if n_pass > 0:
                sr_pass = sub[sub["R2"] > sub["R2_shuffle"]]["Skewness_Ratio"].mean()
            else:
                sr_pass = float("nan")
            sr_all = sub["Skewness_Ratio"].mean()
            print(f"    {cond:<12}: {n_pass}/{n_total} sessions pass R2 gate "
                  f"(SR_pass={sr_pass:.4f}, SR_all={sr_all:.4f})")

        # Barplot
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        melted_sr = []
        for _, row in cebra_lie_pos_df.iterrows():
            melted_sr.append({"Condition": row["Condition"], "Type": "True",
                              "Skewness_Ratio": row["Skewness_Ratio"]})
            melted_sr.append({"Condition": row["Condition"], "Type": "Shuffle",
                              "Skewness_Ratio": row["SR_shuffle"]})
        sns.barplot(data=pd.DataFrame(melted_sr), x="Condition", y="Skewness_Ratio",
                    hue="Type", ax=axes[0],
                    palette={"True": "#440154", "Shuffle": "#B2B2B2"}, errorbar="se")
        axes[0].set_title("Skewness Ratio: True vs Shuffle (Position)")

        melted_r2 = []
        for _, row in cebra_lie_pos_df.iterrows():
            melted_r2.append({"Condition": row["Condition"], "Type": "True", "R2": row["R2"]})
            melted_r2.append({"Condition": row["Condition"], "Type": "Shuffle", "R2": row["R2_shuffle"]})
        sns.barplot(data=pd.DataFrame(melted_r2), x="Condition", y="R2", hue="Type",
                    ax=axes[1], palette={"True": "#440154", "Shuffle": "#B2B2B2"}, errorbar="se")
        axes[1].set_title("R2: True vs Shuffle (Position)")
        plt.suptitle(f"{SUBJECT}: CEBRA-Embedded Lie Algebra -- Position-Driven (per-epoch + shuffle)",
                     fontsize=13, y=1.02)
        plt.tight_layout(); plt.show()

else:
    print("Skipped: CEBRA not installed.")


In [ ]:
# --- GPU cleanup ---
import gc, torch
gc.collect(); torch.cuda.empty_cache()
print("GPU memory cleared.")

## 2. CEBRA Manifold Analysis

CEBRA manifold visualization + statistical validation (InfoNCE loss, k-NN decoding R$^2$, shuffle test). Dimension controlled by `CEBRA_DIM` in Cell 1.

**Note:** May consume significant GPU memory. `gc.collect()` + `torch.cuda.empty_cache()` are called after each section.

### 2.1 Velocity-Driven

In [ ]:
# ==========================================
# CEBRA Manifold HTML Visualization (Plotly) — Velocity-Driven
# ==========================================
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

try:
    import cebra
    from cebra import CEBRA
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    HAS_CEBRA = True
    print("CEBRA is available")
except ImportError:
    HAS_CEBRA = False
    print("CEBRA not installed. This cell requires: pip install cebra")

if HAS_CEBRA:

    def run_cebra_velocity(n_data_list, f_data_list, n_hs0, dt=0.005):
        """Train CEBRA with signed Velocity_x as labels."""
        results = {}
        for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
            neural_list, label_list, sid_list = [], [], []
            for s_idx, (n_raw, f_df) in enumerate(zip(n_data_list, f_data_list)):
                epochs_n, epochs_l, n_ep = extract_epochs(
                    n_raw, f_df, val, dt, label_col="Velocity_x")
                if n_ep < 1:
                    continue
                curr_n = np.concatenate(epochs_n, axis=0)
                curr_l = np.concatenate(epochs_l, axis=0).reshape(-1, 1)
                neural_list.append(curr_n)
                label_list.append(curr_l)
                sid_list.append(np.full(len(curr_l), s_idx))
            if not neural_list:
                continue
            print(f"  {label}: {len(neural_list)} sessions")
            model = CEBRA(
                model_architecture=CEBRA_ARCH, output_dimension=CEBRA_DIM,
                max_iterations=3000, batch_size=512, learning_rate=3e-4,
                temperature=1.5, distance=CEBRA_DISTANCE,
                conditional="time_delta", device="cuda", verbose=True
            )
            model.fit(neural_list, label_list)
            embeddings_list = []
            for i, curr_n in enumerate(neural_list):
                embeddings_list.append(model.transform(curr_n, session_id=i))
            results[label] = (
                np.concatenate(embeddings_list, axis=0),
                np.concatenate(label_list).flatten(),
                np.concatenate(sid_list)
            )
            print(f"  {label} complete.")
        return results

    def visualize_webgl(cebra_res, save_dir, driver_name, downsample_factor=10):
        """Plotly WebGL 3D scatter: Tracking (left) vs Playback (right)."""
        fig = make_subplots(
            rows=1, cols=2,
            specs=[[{"type": "scene"}, {"type": "scene"}]],
            subplot_titles=(f"<b>Active Tracking ({SUBJECT})</b>",
                            f"<b>Passive Playback ({SUBJECT})</b>")
        )
        all_embs = np.concatenate([res[0] for res in cebra_res.values()])
        axis_max = np.max(np.abs(all_embs)) * 1.1
        all_labels = np.concatenate([res[1] for res in cebra_res.values()])
        cmin = float(np.percentile(all_labels, 2))
        cmax = float(np.percentile(all_labels, 98))
        fig.add_trace(
            go.Scatter3d(x=[0, 0], y=[0, 0], z=[0, 0], mode="markers",
                marker=dict(size=0.001, color=[cmin, cmax], colorscale="Plasma",
                    cmin=cmin, cmax=cmax, showscale=True,
                    colorbar=dict(title=driver_name, x=1.05, thickness=15),
                    line=dict(width=0)),
                showlegend=False, hoverinfo="skip"),
            row=1, col=2
        )
        for i, cond in enumerate(["Tracking", "Playback"]):
            if cond not in cebra_res: continue
            emb, labels, sess_ids = cebra_res[cond]
            if downsample_factor > 1:
                emb, labels, sess_ids = emb[::downsample_factor], labels[::downsample_factor], sess_ids[::downsample_factor]
            emb = np.round(emb, 4)
            labels_rounded = np.round(labels, 2)
            for s_id in np.unique(sess_ids):
                mask = (sess_ids == s_id)
                if not np.any(mask): continue
                dynamic_size = 3 + 8 * np.abs(labels[mask]) / max(abs(cmin), abs(cmax), 1e-9)
                fig.add_trace(go.Scatter3d(
                    x=emb[mask, 0], y=emb[mask, 1], z=emb[mask, 2],
                    mode="markers",
                    marker=dict(size=np.round(dynamic_size, 1), line=dict(width=0),
                        color=labels_rounded[mask], colorscale="Plasma",
                        cmin=cmin, cmax=cmax, opacity=0.4, showscale=False),
                    name=f"Sess {int(s_id)}", legendgroup=f"Sess {int(s_id)}",
                    showlegend=bool(i == 0)),
                    row=1, col=i + 1)
        scene_config = dict(
            xaxis=dict(range=[-axis_max, axis_max], visible=False),
            yaxis=dict(range=[-axis_max, axis_max], visible=False),
            zaxis=dict(range=[-axis_max, axis_max], visible=False),
            aspectmode="cube"
        )
        fig.update_layout(
            height=700, title_text=f"Shared Neural Manifold Topology: {SUBJECT}",
            title_x=0.5, scene1=scene_config, scene2=scene_config,
            margin=dict(l=0, r=0, b=0, t=60),
            template="plotly_white", legend=dict(title="Sessions", x=1.1, y=0.5)
        )
        mode_str = "macro" if USE_MACRO_EPOCH else "peri"
        file_name = f"{SUBJECT}_{HS_TAG}_CEBRA_Velocity_{CEBRA_DISTANCE}_{mode_str}.html"
        save_path = os.path.join(save_dir, file_name)
        fig.write_html(save_path, include_plotlyjs='cdn')
        print(f"Saved: {save_path}")

    # ---- Sample sessions, then execute CEBRA ----
    MIN_GC = 10
    N_SAMPLE = 5
    valid_idx = [i for i, nd in enumerate(n_data_all) if nd.shape[0] >= MIN_GC]
    if len(valid_idx) <= N_SAMPLE:
        chosen = valid_idx
    else:
        step = max(1, (len(valid_idx) - 1) // (N_SAMPLE - 1)) if N_SAMPLE > 1 else 0
        chosen = valid_idx[::step][:N_SAMPLE]
    n_subset = [n_data_all[i] for i in chosen]
    f_subset = [f_data_all[i] for i in chosen]
    hs0_subset = sum(1 for i in chosen if i < n_hs0)
    print(f"Sampled {len(chosen)} / {len(valid_idx)} sessions (gc >= {MIN_GC}):")
    for j, i in enumerate(chosen):
        print(f"  {j+1}. session {i}: {n_data_all[i].shape[0]} neurons")

    save_directory = os.path.join(os.getcwd(), "output")
    os.makedirs(save_directory, exist_ok=True)
    print(f"Output: {save_directory}")
    print("=" * 60)
    print(f"{SUBJECT} CEBRA Manifold — Velocity-Driven")
    print("=" * 60)
    cebra_results = run_cebra_velocity(n_subset, f_subset, hs0_subset)
    visualize_webgl(cebra_results, save_directory, "Velocity (signed)")
    print(f"Done! Open {SUBJECT}_{HS_TAG}_CEBRA_Velocity_*.html in a browser.")
else:
    print("Skipped: CEBRA not installed.")

In [ ]:
# --- GPU cleanup ---
import gc, torch
gc.collect(); torch.cuda.empty_cache()
print("GPU memory cleared.")

In [ ]:
# ============================================================
# CEBRA Statistical Validation — Isolated 5-Fold CV (Velocity)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from cebra import CEBRA
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score

K = 5
STEP = 5

n_raw = n_data_all[-1]
f_df = f_data_all[-1]
print(f"Session {len(n_data_all)-1} (last): {n_raw.shape[0]} neurons")

cond_data = {}
for val, cond_name in [(0.0, "Tracking"), (1.0, "Playback")]:
    en, el, n_ep = extract_epochs(n_raw, f_df, val, dt, label_col="Velocity_x", step=STEP)
    chunks_n, chunks_l = [], []
    for e_n, e_l in zip(en, el):
        n = e_n.shape[0]
        chunk_len = n // K
        for k in range(K):
            start = k * chunk_len
            end = n if k == K - 1 else (k + 1) * chunk_len
            chunks_n.append(e_n[start:end].astype(np.float32))
            chunks_l.append(e_l[start:end].astype(np.float32))
    cond_data[cond_name] = {"n": chunks_n, "l": chunks_l}
    print(f"  {cond_name}: {len(chunks_n)} chunks")

all_cv_results = {}
for cond_name in ["Tracking", "Playback"]:
    chunks_n = cond_data[cond_name]["n"]
    chunks_l = cond_data[cond_name]["l"]
    r2_true_folds, r2_shuf_folds = [], []
    loss_true_folds, loss_shuf_folds = [], []

    print(f"\n{'='*60}\n  {cond_name}: {K}-fold CV\n{'='*60}")
    fig_loss, axes_loss = plt.subplots(1, K, figsize=(4*K, 4), sharey=True)
    fig_scatter, axes_scatter = plt.subplots(2, K, figsize=(4*K, 8))

    for fold in range(K):
        train_idx = [i for i in range(K) if i != fold]
        train_n = [chunks_n[i] for i in train_idx]
        train_l = [chunks_l[i] for i in train_idx]
        train_l_shuf = [np.random.permutation(l) for l in train_l]
        X_test, y_test = chunks_n[fold], chunks_l[fold]

        cebra_true = CEBRA(model_architecture=CEBRA_ARCH, output_dimension=CEBRA_DIM,
            max_iterations=3000, batch_size=2048, distance=CEBRA_DISTANCE,
            conditional="time_delta", device="cuda", verbose=True)
        cebra_true.fit(train_n, train_l)
        cebra_shuffle = CEBRA(model_architecture=CEBRA_ARCH, output_dimension=CEBRA_DIM,
            max_iterations=3000, batch_size=2048, distance=CEBRA_DISTANCE,
            conditional="time_delta", device="cuda", verbose=True)
        cebra_shuffle.fit(train_n, train_l_shuf)

        axes_loss[fold].plot(cebra_true.state_dict_['loss'], color='darkred', label='True')
        axes_loss[fold].plot(cebra_shuffle.state_dict_['loss'], color='gray', ls='--', label='Shuffle')
        axes_loss[fold].set_title(f"Fold {fold+1}"); axes_loss[fold].legend(fontsize=6)
        loss_true_folds.append(cebra_true.state_dict_['loss'][-1])
        loss_shuf_folds.append(cebra_shuffle.state_dict_['loss'][-1])

        emb_train = np.concatenate([cebra_true.transform(train_n[i], session_id=0) for i in range(len(train_n))], axis=0)
        emb_test = cebra_true.transform(X_test, session_id=0)
        emb_train_shuf = np.concatenate([cebra_shuffle.transform(train_n[i], session_id=0) for i in range(len(train_n))], axis=0)
        emb_test_shuf = cebra_shuffle.transform(X_test, session_id=0)

        decoder_true = KNeighborsRegressor(n_neighbors=5, weights='distance')
        decoder_shuffle = KNeighborsRegressor(n_neighbors=5, weights='distance')
        decoder_true.fit(emb_train, np.concatenate(train_l))
        decoder_shuffle.fit(emb_train_shuf, np.concatenate(train_l_shuf))

        r2_t = r2_score(y_test, decoder_true.predict(emb_test))
        r2_s = r2_score(y_test, decoder_shuffle.predict(emb_test_shuf))
        r2_true_folds.append(r2_t); r2_shuf_folds.append(r2_s)
        print(f"  Fold {fold+1}: True R2={r2_t:.4f}  Shuffle R2={r2_s:.4f}")

        for row, yp, lbl in [(0, decoder_true.predict(emb_test), "True"), (1, decoder_shuffle.predict(emb_test_shuf), "Shuffle")]:
            ax = axes_scatter[row, fold]
            ax.scatter(y_test[::5], yp[::5], alpha=0.3, s=1, c='darkred' if row==0 else 'gray')
            l = [y_test.min(), y_test.max()]; ax.plot(l, l, '--', color='gray', alpha=0.5)
            ax.set_xlabel("True"); ax.set_ylabel("Pred"); ax.set_title(f"{lbl} Fold {fold+1}")

    plt.suptitle(f"{SUBJECT}: {cond_name} — 5-Fold CV Loss (Velocity)", fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()
    plt.suptitle(f"{SUBJECT}: {cond_name} — 5-Fold CV Predictions (Velocity)", fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()

    all_cv_results[cond_name] = {
        "r2_true": (np.mean(r2_true_folds), np.std(r2_true_folds)),
        "r2_shuf": (np.mean(r2_shuf_folds), np.std(r2_shuf_folds)),
    }

# Report
print(f"\n{'='*60}\n  Isolated 5-Fold CV Report (Velocity)\n{'='*60}")
for cond_name in ["Tracking", "Playback"]:
    r = all_cv_results[cond_name]
    print(f"  {cond_name}: True R2={r['r2_true'][0]:.4f}+/-{r['r2_true'][1]:.4f}  Shuffle R2={r['r2_shuf'][0]:.4f}+/-{r['r2_shuf'][1]:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(4)
bars = [all_cv_results["Tracking"]["r2_true"][0], all_cv_results["Playback"]["r2_true"][0],
        all_cv_results["Tracking"]["r2_shuf"][0], all_cv_results["Playback"]["r2_shuf"][0]]
errs = [all_cv_results["Tracking"]["r2_true"][1], all_cv_results["Playback"]["r2_true"][1],
        all_cv_results["Tracking"]["r2_shuf"][1], all_cv_results["Playback"]["r2_shuf"][1]]
ax.bar(x, bars, yerr=errs, color=['darkred', 'darkblue', 'gray', 'lightgray'], capsize=5)
ax.set_xticks(x); ax.set_xticklabels(['True Tracking', 'True Playback', 'Shuffle Tracking', 'Shuffle Playback'])
ax.set_ylabel("R2 (mean +/- std)")
ax.set_title(f"{SUBJECT}: Isolated 5-Fold CV R2 Summary (Velocity)")
plt.tight_layout(); plt.show()
print(">>> Isolated CV complete.")

In [ ]:
# --- GPU cleanup ---
import gc, torch
del cebra_true, cebra_shuffle
gc.collect(); torch.cuda.empty_cache()
print("GPU memory cleared.")

### 2.2 Position-Driven

In [ ]:
# ============================================================
# Position-Driven CEBRA CEBRA Manifold
# ============================================================
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

try:
    import cebra
    from cebra import CEBRA
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    HAS_CEBRA = True
except ImportError:
    HAS_CEBRA = False

if HAS_CEBRA:
    def run_cebra_position(n_data_list, f_data_list, n_hs0):
        results = {}
        for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
            neural_list, label_list, sid_list = [], [], []
            for s_idx, (n_raw, f_df) in enumerate(zip(n_data_list, f_data_list)):
                epochs_n, epochs_l, n_ep = extract_epochs(n_raw, f_df, val, dt, label_col="Position")
                if n_ep < 1: continue
                neural_list.append(np.concatenate(epochs_n, axis=0))
                label_list.append(np.concatenate(epochs_l, axis=0).reshape(-1, 1))
                sid_list.append(np.full(len(label_list[-1]), s_idx))
            if not neural_list: continue
            model = CEBRA(model_architecture=CEBRA_ARCH, output_dimension=CEBRA_DIM,
                max_iterations=3000, batch_size=512, learning_rate=3e-4,
                temperature=1.5, distance=CEBRA_DISTANCE,
                conditional="time_delta", device="cuda", verbose=True)
            model.fit(neural_list, label_list)
            embeddings_list = [model.transform(n, session_id=i) for i, n in enumerate(neural_list)]
            results[label] = (np.concatenate(embeddings_list, axis=0),
                              np.concatenate(label_list).flatten(),
                              np.concatenate(sid_list))
        return results

    # Sample + run
    MIN_GC, N_SAMPLE = 10, 5
    valid_idx = [i for i, nd in enumerate(n_data_all) if nd.shape[0] >= MIN_GC]
    if len(valid_idx) <= N_SAMPLE:
        chosen = valid_idx
    else:
        step = max(1, (len(valid_idx) - 1) // (N_SAMPLE - 1)) if N_SAMPLE > 1 else 0
        chosen = valid_idx[::step][:N_SAMPLE]
    n_subset = [n_data_all[i] for i in chosen]
    f_subset = [f_data_all[i] for i in chosen]
    hs0_subset = sum(1 for i in chosen if i < n_hs0)
    print(f"Sampled {len(chosen)} / {len(valid_idx)} sessions")

    save_directory = os.path.join(os.getcwd(), "output")
    os.makedirs(save_directory, exist_ok=True)
    cebra_pos_results = run_cebra_position(n_subset, f_subset, hs0_subset)

    # Reuse visualize_webgl from Velocity cell (must be run first)
    visualize_webgl(cebra_pos_results, save_directory, "Position (cm)")

    # Rename to Position-specific
    mode_str = "macro" if USE_MACRO_EPOCH else "peri"
    old_name = f"{SUBJECT}_{HS_TAG}_CEBRA_Velocity_{CEBRA_DISTANCE}_{mode_str}.html"
    new_name = f"{SUBJECT}_{HS_TAG}_CEBRA_Position_{CEBRA_DISTANCE}_{mode_str}.html"
    old_path = os.path.join(save_directory, old_name)
    new_path = os.path.join(save_directory, new_name)
    if os.path.exists(old_path):
        os.rename(old_path, new_path)
        print(f"Saved: {new_path}")
    print("Done!")
else:
    print("Skipped: CEBRA not installed.")

In [ ]:
# --- GPU cleanup ---
import gc, torch
gc.collect(); torch.cuda.empty_cache()
print("GPU memory cleared.")

In [ ]:
# ============================================================
# CEBRA Statistical Validation — Isolated 5-Fold CV (Position)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from cebra import CEBRA
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score

K = 5
STEP = 5

n_raw = n_data_all[-1]
f_df = f_data_all[-1]
print(f"Session {len(n_data_all)-1} (last): {n_raw.shape[0]} neurons")

cond_data = {}
for val, cond_name in [(0.0, "Tracking"), (1.0, "Playback")]:
    en, el, n_ep = extract_epochs(n_raw, f_df, val, dt, label_col="Position", step=STEP)
    chunks_n, chunks_l = [], []
    for e_n, e_l in zip(en, el):
        n = e_n.shape[0]
        chunk_len = n // K
        for k in range(K):
            start = k * chunk_len; end = n if k == K - 1 else (k + 1) * chunk_len
            chunks_n.append(e_n[start:end].astype(np.float32))
            chunks_l.append(e_l[start:end].astype(np.float32))
    cond_data[cond_name] = {"n": chunks_n, "l": chunks_l}
    print(f"  {cond_name}: {len(chunks_n)} chunks")

all_cv_results = {}
for cond_name in ["Tracking", "Playback"]:
    chunks_n = cond_data[cond_name]["n"]
    chunks_l = cond_data[cond_name]["l"]
    r2_true_folds, r2_shuf_folds = [], []
    loss_true_folds, loss_shuf_folds = [], []

    print(f"\n{'='*60}\n  {cond_name}: {K}-fold CV\n{'='*60}")
    fig_loss, axes_loss = plt.subplots(1, K, figsize=(4*K, 4), sharey=True)
    fig_scatter, axes_scatter = plt.subplots(2, K, figsize=(4*K, 8))

    for fold in range(K):
        train_idx = [i for i in range(K) if i != fold]
        train_n = [chunks_n[i] for i in train_idx]
        train_l = [chunks_l[i] for i in train_idx]
        train_l_shuf = [np.random.permutation(l) for l in train_l]
        X_test, y_test = chunks_n[fold], chunks_l[fold]

        cebra_true = CEBRA(model_architecture=CEBRA_ARCH, output_dimension=CEBRA_DIM,
            max_iterations=3000, batch_size=2048, distance=CEBRA_DISTANCE,
            conditional="time_delta", device="cuda", verbose=True)
        cebra_true.fit(train_n, train_l)
        cebra_shuffle = CEBRA(model_architecture=CEBRA_ARCH, output_dimension=CEBRA_DIM,
            max_iterations=3000, batch_size=2048, distance=CEBRA_DISTANCE,
            conditional="time_delta", device="cuda", verbose=True)
        cebra_shuffle.fit(train_n, train_l_shuf)

        axes_loss[fold].plot(cebra_true.state_dict_['loss'], color='darkred', label='True')
        axes_loss[fold].plot(cebra_shuffle.state_dict_['loss'], color='gray', ls='--', label='Shuffle')
        axes_loss[fold].set_title(f"Fold {fold+1}"); axes_loss[fold].legend(fontsize=6)
        loss_true_folds.append(cebra_true.state_dict_['loss'][-1])
        loss_shuf_folds.append(cebra_shuffle.state_dict_['loss'][-1])

        emb_train = np.concatenate([cebra_true.transform(train_n[i], session_id=0) for i in range(len(train_n))], axis=0)
        emb_test = cebra_true.transform(X_test, session_id=0)
        emb_train_shuf = np.concatenate([cebra_shuffle.transform(train_n[i], session_id=0) for i in range(len(train_n))], axis=0)
        emb_test_shuf = cebra_shuffle.transform(X_test, session_id=0)

        decoder_true = KNeighborsRegressor(n_neighbors=5, weights='distance')
        decoder_shuffle = KNeighborsRegressor(n_neighbors=5, weights='distance')
        decoder_true.fit(emb_train, np.concatenate(train_l))
        decoder_shuffle.fit(emb_train_shuf, np.concatenate(train_l_shuf))

        r2_t = r2_score(y_test, decoder_true.predict(emb_test))
        r2_s = r2_score(y_test, decoder_shuffle.predict(emb_test_shuf))
        r2_true_folds.append(r2_t); r2_shuf_folds.append(r2_s)
        print(f"  Fold {fold+1}: True R2={r2_t:.4f}  Shuffle R2={r2_s:.4f}")

        for row, yp, lbl in [(0, decoder_true.predict(emb_test), "True"), (1, decoder_shuffle.predict(emb_test_shuf), "Shuffle")]:
            ax = axes_scatter[row, fold]
            ax.scatter(y_test[::5], yp[::5], alpha=0.3, s=1, c='darkred' if row==0 else 'gray')
            l = [y_test.min(), y_test.max()]; ax.plot(l, l, '--', color='gray', alpha=0.5)
            ax.set_xlabel("True"); ax.set_ylabel("Pred"); ax.set_title(f"{lbl} Fold {fold+1}")

    plt.suptitle(f"{SUBJECT}: {cond_name} — 5-Fold CV Loss (Position)", fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()
    plt.suptitle(f"{SUBJECT}: {cond_name} — 5-Fold CV Predictions (Position)", fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()

    all_cv_results[cond_name] = {
        "r2_true": (np.mean(r2_true_folds), np.std(r2_true_folds)),
        "r2_shuf": (np.mean(r2_shuf_folds), np.std(r2_shuf_folds)),
    }

# Report
print(f"\n{'='*60}\n  Isolated 5-Fold CV Report (Position)\n{'='*60}")
for cond_name in ["Tracking", "Playback"]:
    r = all_cv_results[cond_name]
    print(f"  {cond_name}: True R2={r['r2_true'][0]:.4f}+/-{r['r2_true'][1]:.4f}  Shuffle R2={r['r2_shuf'][0]:.4f}+/-{r['r2_shuf'][1]:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(4)
bars = [all_cv_results["Tracking"]["r2_true"][0], all_cv_results["Playback"]["r2_true"][0],
        all_cv_results["Tracking"]["r2_shuf"][0], all_cv_results["Playback"]["r2_shuf"][0]]
errs = [all_cv_results["Tracking"]["r2_true"][1], all_cv_results["Playback"]["r2_true"][1],
        all_cv_results["Tracking"]["r2_shuf"][1], all_cv_results["Playback"]["r2_shuf"][1]]
ax.bar(x, bars, yerr=errs, color=['darkred', 'darkblue', 'gray', 'lightgray'], capsize=5)
ax.set_xticks(x); ax.set_xticklabels(['True Tracking', 'True Playback', 'Shuffle Tracking', 'Shuffle Playback'])
ax.set_ylabel("R2 (mean +/- std)")
ax.set_title(f"{SUBJECT}: Isolated 5-Fold CV R2 Summary (Position)")
plt.tight_layout(); plt.show()
print(">>> Isolated CV complete.")

In [ ]:
# --- GPU cleanup ---
import gc, torch
del cebra_true, cebra_shuffle
gc.collect(); torch.cuda.empty_cache()
print("GPU memory cleared.")